In [1]:
from scapy.all import *
import struct
from scapy.layers.http import HTTPRequest
from scapy.layers.inet import IP, TCP
from scapy.layers.l2 import Ether
import requests

In [2]:
BASE_URL = 'https://114da01ea829bee03d68cd98-1024-intro-forensics-1.challenge.cscg.live:1337'

In [3]:
bind_layers(TCP, HTTPRequest, dport=1024)

def process_pcap(file_name):
    cap = b'token=unk'

    for (pkt_data, pkt_metadata,) in RawPcapReader(file_name):
        ether_pkt = Ether(pkt_data)

        if 'type' not in ether_pkt.fields:
            # LLC frames will have 'len' instead of 'type'.
            # We disregard those
            continue

        if ether_pkt.type != 0x0800:
            # disregard non-IPv4 packets
            continue

        ip_pkt = ether_pkt[IP]
        if ip_pkt.proto != 6:
            # Ignore non-TCP packet
            continue

        tcp_pkt = ip_pkt[TCP]
        
        if tcp_pkt.dport != 1024:
            continue

        http_pkt = tcp_pkt[HTTPRequest]

        if http_pkt.Method != b'POST':
            continue

        if http_pkt.Path != b'/login':
            continue
        
        cap = http_pkt[Raw].load

    return cap.split(b'=')[1].decode()

cookie_required = process_pcap('./intro-forensics-1.pcapng')
cookie_required

'0bf77fce4af7f09d7937b59b5dfe8ce4c018ea14cd3b363d12ddc7c670ca045313aa6156b40273390e43e6128d32b993742f09d1cea1db3e3837f6082d3e6932'

In [20]:
print(requests.get(f'{BASE_URL}/', cookies={'token': cookie_required}).text)


Welcome to the CSCG Flag Service serving some flags: CSCG{sn00py_sn00p_w1th_w1resh4rk!}

